# Task 1 · Part 2 — From the LLM to the classifier

**Guiding question:** Can Peruvian news headlines be transformed into a reliable news classifier?

Building upon the dataset constructed in Part 1, this notebook compares three increasingly specialized approaches to headline classification: a zero-shot model that requires no training, a general-purpose large language model (LLM) used as a weak annotator, and a lightweight Transformer fine-tuned on the collected headlines. Accordingly, the notebook proceeds from the least to the most task-specific approach, so that each stage can be assessed against the preceding one.

**Methodological note on the reference labels.** Throughout the notebook, the section-based labels assigned in Part 1 (`etiqueta`) serve as the reference labels, given that they derive from the editorial decisions of El Comercio rather than from an automated procedure. The LLM, by contrast, is treated as a weak annotator whose output is audited and evaluated against these reference labels.

In [7]:
import os
import json
import time
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from google import genai
from google.genai import types
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import pipeline

This cell loads the libraries required throughout the notebook. Specifically, `transformers` provides access to pretrained language models, whereas `google-genai` connects to the Gemini LLM; in turn, `dotenv` reads the API key from a local file that is excluded from the repository, thereby preventing credentials from being exposed. Furthermore, `scikit-learn` supplies the stratified data split and the evaluation metrics, while `seaborn` and `matplotlib` render the confusion matrix.

In [8]:
# =============================================
#  CONFIGURATION
# =============================================

# The API key is read from the local .env file, which Git ignores.
load_dotenv()

# Fixed seed so that every random step can be reproduced exactly.
semilla = 42

categorias = ["politica", "economia", "deportes", "ciencia_tecnologia"]

noticias = pd.read_csv("datos/noticias.csv")
print(noticias.shape)
noticias["etiqueta"].value_counts()

(383, 7)


etiqueta
economia              99
deportes              98
ciencia_tecnologia    95
politica              91
Name: count, dtype: int64

The configuration cell loads the API key from the `.env` file, defines a fixed random seed (`42`) to guarantee the reproducibility of every stochastic procedure, and lists the four official categories. Subsequently, the dataset exported in Part 1 is loaded; its dimensions and category distribution confirm that the 383 news items are available for the classification stage.

## 1. Zero-shot baseline

Before any training, a zero-shot classifier establishes the minimum performance that subsequent approaches must exceed. Such a model, pretrained on natural language inference, evaluates how plausibly a headline entails a hypothesis of the form *"This news item is about {category}"*, and thus assigns categories it was never explicitly trained on. Following the assignment, 40 headlines are evaluated; to preserve the balance between classes, ten headlines are sampled from each category.

In [9]:
# Ten headlines per category, sampled with the fixed seed.
muestra_zero_shot = noticias.groupby("etiqueta").sample(n=10, random_state=semilla)
muestra_zero_shot["etiqueta"].value_counts()

etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64

A stratified sample of 40 headlines is drawn, with exactly ten items per category. Stratification prevents the baseline from being distorted by an over-representation of any single category, while the fixed seed ensures that the same sample is obtained in every execution.

In [10]:
# Spanish model trained on natural language inference (lighter than XLM-RoBERTa-large).
clasificador_zero_shot = pipeline(
    "zero-shot-classification",
    model="Recognai/bert-base-spanish-wwm-cased-xnli",
)

# The model reads Spanish, so the candidate labels are written in Spanish.
etiquetas_candidatas = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia y tecnología": "ciencia_tecnologia",
}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The baseline relies on `Recognai/bert-base-spanish-wwm-cased-xnli`, one of the two models suggested by the assignment. It was preferred over `joeddav/xlm-roberta-large-xnli` because it is trained specifically on Spanish and is considerably lighter, which makes it feasible to run on a standard CPU. Since the model processes Spanish text, the candidate labels are expressed in Spanish and subsequently mapped back onto the official category names.

In [11]:
predicciones_zero_shot = []

for titular in muestra_zero_shot["titular"]:
    resultado = clasificador_zero_shot(
        titular,
        candidate_labels=list(etiquetas_candidatas.keys()),
        hypothesis_template="Esta noticia trata sobre {}.",
    )
    # The first label returned is the one with the highest score.
    predicciones_zero_shot.append(etiquetas_candidatas[resultado["labels"][0]])

muestra_zero_shot["prediccion_zero_shot"] = predicciones_zero_shot

accuracy_zero_shot = accuracy_score(muestra_zero_shot["etiqueta"], muestra_zero_shot["prediccion_zero_shot"])
f1_zero_shot = f1_score(muestra_zero_shot["etiqueta"], muestra_zero_shot["prediccion_zero_shot"], average="macro")

print(f"Zero-shot accuracy : {accuracy_zero_shot:.2f}")
print(f"Zero-shot macro F1 : {f1_zero_shot:.2f}")

Zero-shot accuracy : 0.62
Zero-shot macro F1 : 0.59


Without any task-specific training, the zero-shot model reaches an accuracy of 0.62 and a macro F1 of 0.59 on the 40 sampled headlines. Although this performance clearly exceeds the 0.25 expected from random guessing across four categories, it remains modest: roughly four out of ten headlines are misclassified. This limitation is consistent with the nature of the task, since headlines are short and frequently rely on proper names or context-dependent references rather than on explicit thematic vocabulary. Accordingly, a macro F1 of 0.59 constitutes the baseline that the subsequent approaches must exceed.

## 2. Weak supervision with an LLM

In the second stage, a general-purpose LLM is employed as a weak annotator for the entire dataset. Rather than relying on manual annotation, which is costly, the LLM is guided by a structured prompt that defines each category explicitly and requests a machine-readable output. To reduce the number of requests and remain within the free usage tier, headlines are submitted in batches of twenty.

The prompt is structured in three components. First, it assigns the model a role (an expert news editor); second, it defines each category through representative topics, which reduces ambiguity at the boundaries between classes; third, it prescribes a strict JSON output that includes a confidence score. This structure ensures that the responses can be parsed automatically and aligned with the original headlines through their identifiers.

In [12]:
cliente_gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# The lighter Flash-Lite model was used because the standard Flash model was saturated.
modelo_llm = "gemini-flash-lite-latest"

instrucciones = """You are an expert news editor. Classify each Peruvian news headline into exactly one of these four categories:
- politica: government, Congress, elections, political parties, public institutions, justice.
- economia: business, markets, prices, employment, trade, public finance, personal finance.
- deportes: sports, athletes, teams, competitions, sports events.
- ciencia_tecnologia: science, technology, artificial intelligence, digital platforms, environment, scientific research.

Return ONLY a JSON list with one object per headline, in the same order, using this format:
[{"id": 0, "etiqueta": "politica", "confianza": 0.95}]
where "confianza" is your confidence between 0 and 1."""

The prompt is structured in three components. First, it assigns the model a role (an expert news editor); second, it defines each category through representative topics, which reduces ambiguity at the boundaries between classes; third, it prescribes a strict JSON output that includes a confidence score. This structure ensures that the responses can be parsed automatically and aligned with the original headlines through their identifiers. Notably, the Flash-Lite version of Gemini was adopted after the standard Flash model repeatedly returned `503 UNAVAILABLE` errors due to high demand.

In [13]:
lista_etiquetas_llm = []
lotes_fallidos = []
tamano_lote = 20

for inicio in range(0, len(noticias), tamano_lote):
    lote = noticias.iloc[inicio:inicio + tamano_lote]

    titulares_numerados = ""
    for posicion, titular in enumerate(lote["titular"]):
        titulares_numerados += f"{posicion}. {titular}\n"

    # Up to four attempts, since the free tier may be temporarily saturated.
    resultados = []
    for intento in range(4):
        try:
            respuesta = cliente_gemini.models.generate_content(
                model=modelo_llm,
                contents=instrucciones + "\n\nHeadlines:\n" + titulares_numerados,
                config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0),
            )
            resultados = json.loads(respuesta.text)
            break
        except Exception as error:
            print("Retrying batch", inicio, "->", str(error)[:80])
            time.sleep(30 * (intento + 1))

    # Failed batches are recorded instead of being silently skipped.
    if len(resultados) == 0:
        lotes_fallidos.append(inicio)

    for resultado in resultados:
        fila = lote.iloc[resultado["id"]]
        lista_etiquetas_llm.append({
            "indice": fila.name,
            "titular": fila["titular"],
            "etiqueta_llm": resultado["etiqueta"],
            "confianza": resultado["confianza"],
            "modelo_llm": modelo_llm,
            "fecha": date.today().isoformat(),
        })

    print(inicio + len(lote), "headlines processed")
    # Pause between batches to respect the free-tier rate limit.
    time.sleep(7)

print("Failed batches:", lotes_fallidos)

20 headlines processed
40 headlines processed
60 headlines processed
80 headlines processed
100 headlines processed
120 headlines processed
140 headlines processed
160 headlines processed
180 headlines processed
200 headlines processed
220 headlines processed
240 headlines processed
260 headlines processed
280 headlines processed
300 headlines processed
320 headlines processed
340 headlines processed
360 headlines processed
380 headlines processed
383 headlines processed
Failed batches: []


The loop submits the headlines in batches of twenty, each numbered so that every response can be matched to its original headline. Setting the temperature to zero makes the model's answers as deterministic as possible, which favors reproducibility. Furthermore, each batch is attempted up to four times, with progressively longer waits, and any batch that ultimately fails is recorded rather than silently discarded. A pause also separates consecutive requests in order to comply with the rate limits of the free tier. Finally, each result is stored together with the model name and the labeling date, as required by the assignment.

In [14]:
etiquetas_llm = pd.DataFrame(lista_etiquetas_llm)

# Labels outside the four official categories are treated as invalid.
etiquetas_llm.loc[~etiquetas_llm["etiqueta_llm"].isin(categorias), "etiqueta_llm"] = None

etiquetas_llm[["titular", "etiqueta_llm", "confianza", "modelo_llm", "fecha"]].to_csv(
    "datos/etiquetas_llm.csv", index=False
)

print(etiquetas_llm.shape)
print("Invalid labels:", etiquetas_llm["etiqueta_llm"].isna().sum())
etiquetas_llm["etiqueta_llm"].value_counts()

(383, 6)
Invalid labels: 0


etiqueta_llm
politica              103
deportes               98
ciencia_tecnologia     92
economia               90
Name: count, dtype: int64

The LLM annotations are gathered into a DataFrame, and any label falling outside the four official categories is flagged as invalid rather than silently accepted. The file `datos/etiquetas_llm.csv` is then exported with the five columns required by the assignment: `titular`, `etiqueta_llm`, `confianza`, `modelo_llm` and `fecha`.

### 2.1 Human audit

Weak labels are only useful insofar as they are reliable. Accordingly, a random sample of twenty LLM predictions is reviewed manually, and the proportion of labels judged correct by the author is reported as the human–LLM agreement rate.

In [16]:
# Show the complete text of each headline instead of truncating it.
pd.set_option("display.max_colwidth", None)

# Twenty random LLM predictions, sampled with the fixed seed.
muestra_auditoria = etiquetas_llm.sample(n=20, random_state=semilla)
muestra_auditoria[["titular", "etiqueta_llm", "confianza"]].reset_index(drop=True)

,titular,etiqueta_llm,confianza
0,"“El día de mañana me encantaría ver a tantas niñas entrando al automovilismo”: Annia Cillóniz, líder en la categoría S1600, sobre sus sueños en Caminos del Inca",deportes,0.98
1,"Partidos de hoy, sábado 17 de octubre del 2026: hora, canales de televisión y dónde ver fútbol EN VIVO",deportes,0.99
2,"Google lanza Gemini 4 Argon, su nuevo modelo de frontera que supera a GPT-6 Astra y Claude Opus 5.5",ciencia_tecnologia,0.99
3,Científicos descubren cómo predecir cuándo una inteligencia artificial dará respuestas peligrosas,ciencia_tecnologia,0.99
4,Los distritos claves para López Aliaga y Allison: el mapa de Lima entre la dispersión del voto y el arrastre inverso,politica,0.93
5,Noruega estudia prohibir temporalmente las gafas inteligentes,ciencia_tecnologia,0.98
6,Lourdes Alcorta tras suspensión de Comisión de Constitución: “Están buscando es cualquier situación para encrispar el ambiente”,politica,0.99
7,Wall Street abre mixto arrastrado por los fabricantes de semiconductores,economia,0.95
8,"Las empresas de IA son incapaces de controlar sus modelos, alerta un exinvestigador de OpenAI",ciencia_tecnologia,0.99
9,¡Entre los ocho mejores! Angelo Caro clasificó a la final del World Skate Games Asunción 2026,deportes,0.99


In [17]:
# Author's manual verdict for each audited prediction, in table order (True = correct).
revision_humana = [
    True, True, True, True, True,   # 0-4
    True, True, True, True, True,   # 5-9
    True, True, True, True, True,   # 10-14
    True, True, True, True, True,   # 15-19
]

aciertos_auditoria = sum(revision_humana)
print(f"Human-LLM agreement: {aciertos_auditoria}/20 = {aciertos_auditoria / 20:.0%}")

Human-LLM agreement: 20/20 = 100%


Each of the twenty predictions was examined individually by the author and recorded as correct or incorrect. All twenty labels were judged correct, yielding a human–LLM agreement of 20/20 (100%). Nevertheless, several items lie close to the boundaries between categories, and their assignment depends on the criterion adopted:

- **Item 5** (Norway considering a ban on smart glasses) is correctly labelled as `ciencia_tecnologia` when classified by its main subject, although its regulatory angle could also suggest a public-policy reading.
- **Item 15** (taxes on Temu and similar platforms) fits `economia`, since its focus lies on taxation and economic impact; however, had the headline centered on a new state regulation, `politica` would have been a plausible alternative.
- **Item 16** (ProInversión and Lima's Metro lines) is consistent with `economia`, given its emphasis on investment and project tenders rather than on the governmental decision itself.
- **Item 17** (how brands can appear in ChatGPT) could arguably be read as economic news on marketing, although its focus on how a technology operates justifies its assignment to `ciencia_tecnologia`.

These cases suggest that the boundaries between categories should be defined explicitly: `deportes` covers competitions, athletes, teams, results and sports broadcasts; `ciencia_tecnologia`, artificial intelligence, software, devices, innovation and scientific research; `politica`, elections, parties, authorities, political institutions and government decisions; and `economia`, financial markets, companies, inflation, GDP, taxes, investment and economic activity.

Finally, it should be noted that the LLM's high confidence scores (0.93–0.99) do not by themselves guarantee correctness, since they merely reflect the model's own estimate of certainty. Accordingly, the 100% agreement rate should be interpreted with caution, given the small size of the audited sample and the absence of deliberately ambiguous headlines within it.

In [18]:
# Attach the LLM label to each news item through its original index.
noticias = noticias.join(etiquetas_llm.set_index("indice")[["etiqueta_llm", "confianza"]])

acuerdo_referencia = (noticias["etiqueta"] == noticias["etiqueta_llm"]).mean()
print(f"Agreement between LLM labels and section-based labels: {acuerdo_referencia:.0%}")

pd.crosstab(noticias["etiqueta"], noticias["etiqueta_llm"], rownames=["Section label"], colnames=["LLM label"])

Agreement between LLM labels and section-based labels: 91%


LLM label,ciencia_tecnologia,deportes,economia,politica
Section label,,,,
ciencia_tecnologia,85,0,4,6
deportes,0,98,0,0
economia,7,0,81,11
politica,0,0,5,86


Beyond the human audit, the LLM labels are compared with the section-based reference labels across the entire dataset. Overall, both criteria coincide in 91% of cases (350 of 383 news items). The agreement is perfect for sports (98 of 98), whose vocabulary is highly distinctive. By contrast, the main divergence lies at the boundary between economy and politics: eleven articles published in the economy section were labelled as `politica` by the LLM, and five politics articles were labelled as `economia`. Similarly, a smaller number of science and technology articles were assigned to politics (6) or economy (4). These discrepancies do not necessarily indicate LLM errors; rather, they reflect that the reference label captures the section in which El Comercio chose to publish an article, whereas the LLM judges each headline solely by its content. Consequently, topics such as taxation, public budgets or technology regulation, which inherently span more than one category, account for most of the disagreements.

## 3. Data split (70 / 15 / 15)

To obtain an unbiased estimate of performance, the dataset is divided into three disjoint subsets: a training set (70%) from which the model learns, a validation set (15%) used to monitor training, and a test set (15%) that remains strictly isolated until the final evaluation. Both splits are stratified by category and use the fixed seed, so that each subset preserves the class distribution and the partition can be reproduced exactly.

In [19]:
# First split: 70% training and 30% temporary set.
entrenamiento, temporal = train_test_split(
    noticias, test_size=0.30, stratify=noticias["etiqueta"], random_state=semilla
)

# Second split: the temporary set is divided equally into validation and test.
validacion, prueba = train_test_split(
    temporal, test_size=0.50, stratify=temporal["etiqueta"], random_state=semilla
)

print("Training   :", entrenamiento.shape[0])
print("Validation :", validacion.shape[0])
print("Test       :", prueba.shape[0])

Training   : 268
Validation : 57
Test       : 58


The partition is performed in two steps: first, 30% of the data is set aside; subsequently, this portion is divided equally into validation and test sets. Stratifying both steps by the reference label guarantees that the four categories are represented proportionally in every subset. From this point onwards, the test set is not used for any decision until the final evaluation, thereby preventing the inflated metrics that arise from evaluating a model on data it has already seen.